# Objectives
Fraud Detection Machine Learning Pipeline (using Amazon SageMaker). The followig services are used in the project:

* **Amazon SageMaker Pipelines** → orchestration
* **SageMaker Training Jobs** → XGBoost training
* **SageMaker Processing Jobs** → preprocessing/evaluation
* **SageMaker Model Registry** → versioned models
* **CodePipeline + CodeBuild** → CI/CD
* **SageMaker Endpoint** → production inference

# Directory Structure

In [ ]:
%%writefile README.md
# Project Structure
fraud-sagemaker-mlops/
│
├── README.md
├── requirements.txt
│
├── config.py
│
├── pipeline.py
│
├── scripts/
│   │
│   ├── preprocess.py
│   ├── train.py
│   ├── evaluate.py
│   └── inference.py
│
├── deployment/
│   └── deploy_endpoint.py
│
└── data/
    └── raw
        └── fraud_transactions_dataset.csv

In [ ]:
# ---------------------
# Create the Directory Structure
# ---------------------
!mkdir data scripts deployment

# Configuration

In [ ]:
%%writefile config.py
AWS_REGION = "eu-east-1"

ROLE_ARN = (
    "arn:aws:iam::471112765184:role/cfst-4216-8cb00ae83d298f5cd2-SageMakerExecutionRole-6mrByY9qoGOL"
)

BUCKET = (
    "sagemaker-us-east-1-905418372687"
)

PIPELINE_NAME = (
    "fraud-detection-xgboost-pipeline-1"
)

MODEL_PACKAGE_GROUP = (
    "fraud-detection-models"
)

ENDPOINT_NAME = (
    "fraud-detection-production"
)

# requirements.txt

In [ ]:
%%writefile requirements.txt
sagemaker==2.248.0
boto3
pandas
numpy
scikit-learn
xgboost

In [ ]:
!pip install -r requirements.txt

# Verify Installation

In [ ]:
# Check SageMaker Version
from importlib.metadata import version

print(version("sagemaker"))

# or
!pip show sagemaker

# Preprocessing

In [ ]:
%%writefile scripts/preprocess.py
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    roc_auc_score
)

from xgboost import XGBClassifier

import joblib


In [ ]:
%%writefile -a scripts/preprocess.py
# -----------------------------
# 1. Load Dataset
# -----------------------------
Data_path = "dataset/raw/fraud_transactions_dataset.csv"
df = pd.read_csv(Data_path)

print("Dataset Preview:")
print(df.head())


In [ ]:
%%writefile -a scripts/preprocess.py
import pandas as pd
import argparse
import os

def preprocess(input_file, output_dir):
    df = pd.read_csv(input_file)
    
    df["transaction_time"] = (
        pd.to_datetime(df["transaction_time"])
    )

    df["hour"] = (
        df.transaction_time.dt.hour
    )

    df["day"] = (
        df.transaction_time.dt.day
    )

    df["month"] = (
        df.transaction_time.dt.month
    )

    df.drop(
        [
            "transaction_id",
            "customer_id",
            "transaction_time"
        ],
        axis=1,
        inplace=True
    )

    df["card_present"] = (
        df.card_present
        .map(
            {
                "Yes":1,
                "No":0
            }
        )
    )

    df["previous_fraud"] = (
        df.previous_fraud
        .map(
            {
                "Yes":1,
                "No":0
            }
        )
    )

    df = pd.get_dummies(
        df,
        columns=[
            "merchant_category",
            "transaction_country"
        ]
    )

    os.makedirs(
        output_dir,
        exist_ok=True
    )

    df.to_csv(
        f"{output_dir}/train.csv",
        index=False
    )

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument(
        "--input"
    )
    parser.add_argument(
        "--output"
    )
    args = parser.parse_args()
    preprocess(
        args.input,
        args.output
    )

# TODO
- cleaning
- splitting

# Train

In [ ]:
%%writefile scripts/train.py

import argparse
import pandas as pd
import xgboost as xgb
import os

def train(
    train_path,
    model_path
):

    df = pd.read_csv(
        train_path
    )

    X = df.drop(
        "fraud",
        axis=1
    )

    y = df.fraud

    model = xgb.XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="auc",
        scale_pos_weight=5
    )

    model.fit(
        X,
        y
    )

    os.makedirs(
        model_path,
        exist_ok=True
    )

    model.save_model(
        f"{model_path}/model.json"
    )

# if __name__ == "__main__":
#     parser=argparse.ArgumentParser()
#     parser.add_argument(
#         "--train"
#     )
#     parser.add_argument(
#         "--model-dir"
#     )
#     args=parser.parse_args()
#     train(
#         args.train,
#         args.model_dir
#     )

# Evaluate

In [ ]:
%%writefile scripts/evaluate.py

import json
import pandas as pd
import xgboost as xgb

from sklearn.metrics import (
    roc_auc_score,
    precision_score,
    recall_score
)

model=xgb.XGBClassifier()

model.load_model(
    "/opt/ml/processing/model/model.json"
)

df=pd.read_csv(
    "/opt/ml/processing/test/test.csv"
)

X=df.drop(
    "fraud",
    axis=1
)

y=df.fraud

prediction=model.predict(
    X
)

probability=model.predict_proba(
    X
)[:,1]

metrics={
    "auc":
    roc_auc_score(
        y,
        probability
    ),

    "precision":
    precision_score(
        y,
        prediction
    ),

    "recall":
    recall_score(
        y,
        prediction
    )
}

with open(
    "/opt/ml/processing/evaluation.json",
    "w"
) as f:

    json.dump(
        metrics,
        f
    )

# SageMaker Pipeline

In [ ]:
%%writefile pipeline.py

import sagemaker

from sagemaker.workflow.pipeline import Pipeline

from sagemaker.workflow.steps import (
    ProcessingStep,
    TrainingStep
)

from sagemaker.processing import ScriptProcessor

from sagemaker.estimator import Estimator

from sagemaker.inputs import TrainingInput

import config

session = sagemaker.Session()

processor = ScriptProcessor(

    image_uri=
    sagemaker.image_uris.retrieve(
        framework="sklearn",
        region=session.boto_region_name,
        version="1.2-1"
    ),

    command=[
        "python3"
    ],

    role=config.ROLE_ARN,

    instance_count=1,

    instance_type="ml.m5.large"

)

processing_step = ProcessingStep(
    name="FraudDataProcessing",
    processor=processor,
    code=
    "scripts/preprocess.py"
)

xgb_estimator = Estimator(
    image_uri=
    sagemaker.image_uris.retrieve(
        framework="xgboost",
        region=session.boto_region_name,
        version="1.7-1"
    ),

    role=config.ROLE_ARN,
    instance_count=1,
    instance_type="ml.m5.large",
    output_path=
    f"s3://{config.BUCKET}/models"
)

training_step = TrainingStep(
    name="FraudXGBoostTraining",
    estimator=xgb_estimator,
    inputs={
        "train":
        TrainingInput(
            s3_data=
            f"s3://{config.BUCKET}/processed/"
        )
    }
)

pipeline = Pipeline(
    name=config.PIPELINE_NAME,
    steps=[
        processing_step,
        training_step
    ]
)

pipeline.upsert(
    role_arn=config.ROLE_ARN
)

print(
"Pipeline created"
)

# Run the pipeline
Inside SageMaker Studio, run:

In [ ]:
!python pipeline.py

# Verify if the pipeline is created

Open AWS Console -> SageMaker Studio -> Pipelines -> fraud-detection-xgboost-pipeline

Now you can execute it by clicking "Execute Latest Version" or you can schedule its execution.


# Closing
Congratulations! This is a key milestone that concludes the pipeline creation in SageMaker! 

---

# Deploy Endpoint

In [ ]:
%%writefile deployment/deploy_endpoint.py

import boto3
from config import ENDPOINT_NAME

sm = boto3.client(
    "sagemaker"
)

model_name = (
    "fraud-xgb-approved-model"
)

sm.create_endpoint_config(

    EndpointConfigName=
    "fraud-endpoint-config",
    ProductionVariants=[
        {
        "VariantName":"primary",
        "ModelName":model_name,
        "InitialInstanceCount":1,
        "InstanceType":
        "ml.m5.large",
        "InitialVariantWeight":1
        }
    ]
)

sm.create_endpoint(
    EndpointName=
    config.ENDPOINT_NAME,
    EndpointConfigName=
    "fraud-endpoint-config"
)

print(
"Endpoint deployment started"
)

In [ ]:
!python deployment/deploy_endpoint.py 

# Test Endpoint

# Perform Inference

In [ ]:
!python scripts/inference.py

In [ ]:
%%writefile scripts/inference.py

import boto3

runtime=boto3.client(
    "sagemaker-runtime"
)

response = runtime.invoke_endpoint(
    EndpointName=
    "fraud-detection-production",
    ContentType=
    "text/csv",
    Body=
    "8500,1,0,1,2,15,7"
)

print(
response["Body"].read()
)